In [1]:
import os
import requests
import pandas as pd
from io import StringIO
from Bio import SeqIO

COG_data_dir = '/active-data/datasets/COG_data'
os.makedirs(COG_data_dir, exist_ok=True)

if os.path.isfile(f'{COG_data_dir}/cog-24.def.tsv') and os.path.isfile(f'{COG_data_dir}/cog-24.fun.tsv'):
    COG_table = pd.read_csv(f'{COG_data_dir}/cog-24.def.tsv', sep='\t')
    COG_info = pd.read_csv(f'{COG_data_dir}/cog-24.fun.tsv', sep='\t')
else:
    url = r'https://ftp.ncbi.nlm.nih.gov/pub/COG/COG2024/data/cog-24.def.tab'
    response = requests.get(url)
    COG_table = pd.read_table(StringIO(response.text), header = None, names = ['COG_ID', 'Functional_Category', 'Gene_Name', 'Product', 'Pathway', 'PMID', 'PDB'])
    
    url = r'https://ftp.ncbi.nlm.nih.gov/pub/COG/COG2024/data/cog-24.fun.tab'
    cog_response = requests.get(url)
    COG_class = {}
    COG_info = pd.DataFrame()
    for line in cog_response.text.split('\n'):
        info_l = line.split('\t')
        if len(info_l) == 2:
            COG_class[info_l[0]] = info_l[1]
        elif len(info_l) > 2:
            temp_dict = {'char':info_l[0], 'class': COG_class[info_l[1]], 'description': info_l[-1]}
            COG_info = pd.concat([COG_info, pd.DataFrame([temp_dict])], ignore_index = True)
    COG_table.to_csv(f'{COG_data_dir}/cog-24.def.tsv', index=False, sep='\t')
    COG_info.to_csv(f'{COG_data_dir}/cog-24.fun.tsv', index=False, sep='\t')

In [2]:
import ast
import os
import pandas as pd

def extract_genus_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_genus_name)
counts = all_data['genus_clean'].value_counts()
keep_genus = counts[counts >= 400].index.to_list()
print(keep_genus)

['Escherichia', 'Klebsiella', 'Staphylococcus', 'Pseudomonas', 'Bacillus', 'Salmonella', 'Streptococcus', 'Streptomyces', 'Acinetobacter', 'Enterococcus', 'Bordetella', 'Enterobacter', 'Xanthomonas', 'Campylobacter', 'Vibrio', 'Mycobacterium', 'Corynebacterium', 'Burkholderia', 'Listeria', 'Citrobacter', 'Helicobacter']


In [3]:
from tqdm import tqdm
import numpy as np

head = 'query	seed_ortholog	evalue	score	eggNOG_OGs	max_annot_lvl	COG_category	Description	Preferred_name	GOs	EC	KEGG_ko	KEGG_Pathway	KEGG_Module	KEGG_Reaction	KEGG_rclass	BRITE	KEGG_TC	CAZy	BiGG_Reaction	PFAMs'.split('\t')
cog2cat = dict(zip(COG_table["COG_ID"], COG_table["Functional_Category"]))

def process_ogs(ogs_str):
    if pd.isna(ogs_str) or 'COG' not in ogs_str:
        return ''
    
    og_list = ogs_str.split(',')
    cog_set = set()
    for og in og_list:
        if 'COG' in og:
            cog = og.split('@')[0]
            cog_set.add(cog)
    
    cat_chars = set()
    for cog in cog_set:
        if cog in cog2cat:
            cat_chars.update(cog2cat[cog])
    
    return ''.join(sorted(cat_chars))

for genus_name in keep_genus:
    base_folder = f'/active-data/analysis_results/chr_pla/genus'
    os.chdir(f'{base_folder}/statistics_records/{genus_name}')
    replicon_data = pd.read_csv('pseudogene_statistics.tsv', sep='\t')
    rep_COG_data = []
    with tqdm(total = len(replicon_data), desc=f'{genus_name}', leave=True, ncols=100, unit='B', unit_scale=True) as pbar:
        for idx, row in replicon_data.iterrows():
            acc_n, contig = row['contig'].split('-')
            eggnog_dir = f'/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/eggnog_results/{acc_n}'
            eggnog_re = pd.read_csv(f'{eggnog_dir}/{acc_n}.emapper.annotations', comment = '#', sep = '\t', header = None, names = head)
            eggnog_re = eggnog_re[eggnog_re['query'].str.contains(contig)]

            eggnog_re['COG_category_new'] = eggnog_re['eggNOG_OGs'].apply(process_ogs)
            
            mask = (eggnog_re['COG_category_new'] != '') & (eggnog_re['COG_category_new'] != eggnog_re['COG_category'])
            eggnog_re['COG_category_corrected'] = np.where(mask, eggnog_re['COG_category_new'], eggnog_re['COG_category'])
    
            temp_dict = {'accession': row['contig'], 'All coding CDSs': row['CDSs (with protein)'], 'Categorized CDSs': len(eggnog_re[eggnog_re['COG_category_corrected'] != '-'])}
            for idx_COG, row_COG in COG_info.iterrows():
                temp_dict[row_COG['char']] = 0
            for COG_category in eggnog_re['COG_category_corrected']:
                char_list = list(COG_category)
                for char_n in char_list:
                    try:
                        temp_dict[char_n] += 1
                    except:
                        pass
            rep_COG_data.append(pd.DataFrame([temp_dict]))
            pbar.update(1)
    rep_COG_data = pd.concat(rep_COG_data, ignore_index=True)
    rep_COG_data.to_csv('COG_statistics.tsv', sep='\t', index=False)

Escherichia: 100%|██████████████████████████████████████████████| 15.4k/15.4k [11:34<00:00, 22.2B/s]
Klebsiella: 100%|███████████████████████████████████████████████| 15.0k/15.0k [11:04<00:00, 22.5B/s]
Staphylococcus: 100%|███████████████████████████████████████████| 5.08k/5.08k [02:00<00:00, 42.0B/s]
Pseudomonas: 100%|██████████████████████████████████████████████| 3.14k/3.14k [02:45<00:00, 19.0B/s]
Bacillus: 100%|█████████████████████████████████████████████████| 3.99k/3.99k [02:28<00:00, 26.9B/s]
Salmonella: 100%|███████████████████████████████████████████████| 4.32k/4.32k [03:13<00:00, 22.3B/s]
Streptococcus: 100%|████████████████████████████████████████████| 1.78k/1.78k [00:39<00:00, 45.0B/s]
Streptomyces: 100%|█████████████████████████████████████████████| 2.46k/2.46k [02:15<00:00, 18.2B/s]
Acinetobacter: 100%|████████████████████████████████████████████| 3.74k/3.74k [02:01<00:00, 30.8B/s]
Enterococcus: 100%|█████████████████████████████████████████████| 3.29k/3.29k [01:17<00:00,